###  Introducción al *Parameter-Efficient Fine-Tuning* (PEFT)

Vamos a ver una familia de técnicas denominadas **Parameter-Efficient Fine-Tuning**, a menudo abreviada como **PEFT**. Está estrechamente relacionada con la congelación de parámetros durante el entrenamiento; de hecho, la mayoría de los métodos PEFT implican congelar la gran mayoría o la totalidad de los parámetros originales de un modelo.

No vamos a entrar en detalles exhaustivos, pero como conjunto de técnicas merece la pena conocerlas, especialmente si tienes previsto ajustar modelos muy grandes y no dispones de recursos informáticos masivos.

Ahora bien, estos no son los tipos de métodos necesarios para entrenar modelos pequeños de, digamos, 100 millones o quizás mil millones de parámetros. Estos modelos están realmente diseñados para cuando se trabaja con arquitecturas descomunales del orden de los **100.000 millones de parámetros**, donde aun así sigue siendo cierto que se necesitará al menos una GPU potente, e idealmente varias. Por lo tanto, no es que podamos usar estos trucos inteligentes para afinar un GPT completo en una laptop doméstica.

---

###  Naturaleza y Objetivos del PEFT

Como hemos mencionado, el PEFT no es un único método, sino una familia de técnicas que permiten realizar un ajuste fino. Tampoco es algo que se utilizaría para el preentrenamiento; se trata de técnicas que se aplican a grandes modelos ya preentrenados que se quieren adaptar a una tarea específica.

Aunque esa tarea pueda ser generar nuevo texto (como un chatbot), los métodos PEFT tienden a funcionar mejor para tareas más específicas y acotadas, como la categorización. La idea central es congelar tantos parámetros del modelo como sea posible mientras se entrena únicamente un número muy reducido de ellos.

* **Idea general:** Algunos métodos consisten en insertar nuevas capas entrenables en un modelo congelado que aplican transformaciones a los datos, haciendo que el modelo preentrenado funcione mejor en una tarea específica.
* **Ventaja principal:** Permite realizar un *fine-tuning* avanzado cuando no se tiene acceso a la potencia de cálculo computacional que normalmente se necesitaría para entrenar toda la red desde cero.
* **Desventaja computacional:** Por otra parte, incluso realizar un pase hacia adelante (*forward pass*) con un modelo de 100.000 millones de parámetros puede llevar varios segundos para un puñado de tokens. Por lo tanto, la limitación de recursos a la que se refieren estas técnicas sigue exigiendo un clúster de varias GPUs muy potentes en contraste con cientos o miles de GPUs de gama alta.

---

###  Desventajas y Generalización

Como probablemente puedas imaginar, la principal limitación del PEFT es que es poco probable que, congelando la mayor parte del modelo y añadiendo un par de parámetros extra aquí y allá, obtengas un rendimiento superior al de un modelo completamente entrenado y afinado de extremo a extremo.

La gente suele usar PEFT para entrenar modelos orientados a tareas muy específicas —como categorizar documentos en un dominio concreto—. Eso funciona muy bien para ese nicho particular, pero un modelo entrenado mediante PEFT suele tener dificultades para **generalizar** a nuevas tareas.

Siempre existe un equilibrio: cuanto más concreta y clara sea la tarea, más fácil y satisfactorio será afinar un modelo que en su mayor parte está congelado; sin embargo, esto implica que el modelo será menos capaz de adaptarse o generalizar a otros dominios o propósitos nuevos.

###  Técnicas PEFT: *Adapters* (Adaptadores)

Existen quizás una docena de técnicas específicas que se engloban bajo el paraguas del **PEFT** (*Parameter-Efficient Fine-Tuning*). Vamos a destacar cuatro de ellas para hacernos una idea de cómo funcionan.

**Adapters (Adaptadores):** La idea principal consiste en congelar todo el modelo base e insertar algunas capas adicionales en forma de módulos en puntos estratégicos. Muchos de estos módulos adaptadores (*adapter modules*) —estas pequeñas capas— se sitúan típicamente entre la subcapa de atención y la subcapa MLP, aunque también pueden colocarse entre bloques de transformadores (*transformer blocks*).

El concepto central es que estos pequeños módulos de red neuronal aprendan alguna transformación relevante para la tarea específica sin necesidad de alterar los parámetros ni el conocimiento del mundo que el modelo ya adquirió durante su preentrenamiento.

Curiosamente, estos pequeños módulos añadidos presentan una estructura similar a la de un *autoencoder*: suelen comenzar con una alta dimensionalidad en la entrada y en la salida, pero cuentan con una capa oculta en el medio que reduce drásticamente la dimensionalidad, acompañada de alguna función de activación no lineal como GELU.

###  Técnicas PEFT: Descomposición de Bajo Rango (*LoRA / DoRA*)

Existen otras técnicas que se basan en la descomposición de bajo rango de las matrices de pesos (*low-rank decomposition* y *low-rank weight matrix*). La idea detrás de una aproximación de bajo rango es utilizar una matriz más pequeña que la original —con muchos menos parámetros— que actúe como una representación comprimida o aproximada de la matriz completa.

Imaginemos, por ejemplo, que los pesos de atención tienen una dimensión de $1000 \times 1000$, y supongamos que podemos aproximar dicha matriz utilizando dimensiones reducidas (por ejemplo, descomponiéndola en matrices más pequeñas de $100 \times 100$). Esto significa que, en lugar de tener un millón de parámetros entrenables ($1000 \times 1000$), pasaríamos a manejar solo $10.000$ parámetros entrenables.

Por supuesto, la aproximación de bajo rango no es perfecta: en el proceso se pierde información y precisión. Sin embargo, se confía en que los detalles que se descartene no sean críticos para la tarea específica en la que estamos entrenando el modelo. La gran ventaja es que resulta mucho más sencillo y eficiente entrenar $10.000$ parámetros que un millón.

Si esta operación se repite para múltiples matrices de pesos en todo el modelo, el ahorro acumulado en recursos computacionales llega a ser masivo. Es aquí donde **LoRA** (*Low-Rank Adaptation*) y **DoRA** (*Weight-Decomposed Low-Rank Adaptation*) se posicionan como dos métodos específicos que entran dentro de esta categoría, basando el ajuste en aproximaciones de bajo rango mediante la descomposición de matrices en lugar de modificar las matrices de pesos completas.

###  Técnicas PEFT: Módulo Prefijo (*Prefix Tuning*)

Aquí la idea consiste en congelar todo el modelo base y luego añadir un conjunto de vectores o capas adicionales —conocidos como prefijos— justo antes de los *embeddings* o al inicio del modelo, los cuales se entrenan específicamente para la tarea requerida.

Este módulo prefijo aprende a transformar y guiar los tokens de entrada mediante vectores de *embeddings* modificados que se encuentran optimizados para maximizar el rendimiento en esa tarea específica, sin alterar los pesos del modelo original.

###  Técnicas PEFT: *BitFit*

Por último, el cuarto enfoque consiste en congelar todos los pesos del modelo y entrenar únicamente los términos de sesgo (*bias terms*).

La idea principal es que existe una cantidad relativamente pequeña de términos de sesgo en comparación con el total de pesos, lo que hace que su entrenamiento sea mucho más ligero y eficiente.

Aunque los términos de sesgo pueden ser bastante potentes en los modelos, su impacto se limita principalmente a desplazar las distribuciones; por lo tanto, tienen un efecto menor sobre la remodelación profunda de los espacios en los que se representan los *embeddings*.

## Cuandro comparativo

### Cuadro Comparativo: Técnicas de Parameter-Efficient Fine-Tuning (PEFT)

| Técnica PEFT | ¿En qué consiste? | Ventajas Principales | Desventajas / Limitaciones |
| :--- | :--- | :--- | :--- |
| **Adapters (Adaptadores)** | Inserción de pequeños módulos en forma de *autoencoder* (baja dimensionalidad central) entre las capas de atención y MLP. | • Permite aprender transformaciones específicas sin alterar el conocimiento previo.<br>• Muy eficiente en número de parámetros entrenables. | • Añade pequeñas capas adicionales, lo que puede incrementar ligeramente la latencia en la inferencia. |
| **LoRA / DoRA** | Descomposición de bajo rango (*low-rank decomposition*) de las matrices de pesos mediante matrices más pequeñas comprimidas. | • Reducción masiva de parámetros entrenables (ej. de 1M a 10K).<br>• Excelente relación entre rendimiento y eficiencia de memoria.<br>• Sin latencia adicional en inferencia si se fusionan los pesos. | • Aproximación matemática imperfecta (se pierde cierta precisión fina en comparación con el entrenamiento completo). |
| **Prefix Tuning (Módulos Prefijo)** | Adición de vectores entrenables o prefijos justo antes de los *embeddings* o al inicio del modelo para guiar las entradas. | • Modifica el comportamiento del modelo de forma global desde la entrada sin tocar las capas internas de pesos. | • Consume longitud efectiva de contexto (*sequence length*), ya que los prefijos ocupan espacio en la ventana de tokens. |
| **BitFit** | Congelación de todos los pesos del modelo entrenando únicamente los términos de sesgo (*bias terms*). | • Cantidad mínima de parámetros a entrenar.<br>• Extremadamente ligero en uso de memoria y cómputo. | • Impacto limitado: ideal para desplazar distribuciones, pero muy pobre para remodelar profundamente los espacios de representación. |

###  Otras Técnicas Destacadas bajo el Paraguas de PEFT

Estas no son todas las técnicas de PEFT existentes; también existen muchas otras variantes que podrás encontrar en la literatura especializada. Sin embargo, todas ellas se basan o se relacionan de alguna manera con los principios que ya hemos visto. Entre las más destacadas se encuentran:

* **P-Tuning y P-Tuning v2:** Evoluciones de los módulos prefijo. En lugar de prefijos fijos, utilizan una red neuronal recurrente (LSTM) o reparametrizaciones continuas para generar y optimizar los embeddings de los prompts de manera dinámica en cada capa.
* **IA3 (Infusion by Inhibiting and Amplifying Inner Activations):** Multiplica las activaciones internas del modelo por vectores entrenables aprendidos, escalando selectivamente los canales de atención y de los bloques MLP con una eficiencia de parámetros extrema.
* **QLoRA (Quantized LoRA):** Una variante avanzada de LoRA que combina la descomposición de bajo rango con una cuantización masiva del modelo base a 4 bits, permitiendo afinar modelos gigantescos directamente en hardware de consumo o con recursos muy limitados.

Estos modelos no serán como un *full fine-tuning model*, pero pueden funcionar muy bien si se tiene una tarea que es muy fácil de cuantificar en términos de una función de pérdida, menos abierta y difícil de evaluar, como la generalización de texto sobre una amplia gama de temas.